In [4]:
pip install nltk scikit-learn

In [5]:

import re
from collections import Counter

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score


# --------------------------------------------------
# STEP 1: Prepare labeled training data
# --------------------------------------------------

training_sentences = [

    # Machine Learning
    "Machine learning uses data to make predictions.",
    "Neural networks are used in deep learning.",
    "Artificial intelligence models learn from data.",
    "Regression predicts continuous numerical values.",
    "Classification assigns data to different classes.",
    "Training data helps machine learning algorithms learn.",
    "Supervised learning uses labeled examples.",
    "A neural network contains interconnected neurons.",

    # Cricket
    "The cricket team won the match yesterday.",
    "The player scored a century in the match.",
    "The batsman hit a six over the boundary.",
    "The bowler took three wickets today.",
    "The cricket team played in the tournament.",
    "The captain won the toss before the match.",
    "The batter scored fifty runs.",
    "The umpire declared the batsman out.",

    # Programming
    "Python is a popular programming language.",
    "Python supports object oriented programming.",
    "Java is used to develop software applications.",
    "A programmer writes code using programming languages.",
    "Functions help organize a computer program.",
    "Classes and objects are used in Java.",
    "Software developers debug errors in code.",
    "Python supports functions and data structures."
]


training_labels = (
    ["ML"] * 8
    + ["CRICKET"] * 8
    + ["PROGRAMMING"] * 8
)


# --------------------------------------------------
# STEP 2: Build NLP and ML classifier
# --------------------------------------------------

model = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1, 2))),
    ("classifier", MultinomialNB(alpha=0.1))
])

model.fit(training_sentences, training_labels)


# --------------------------------------------------
# STEP 3: Split continuous text into sentences
# --------------------------------------------------

def split_sentences(text):

    sentences = re.split(r"(?<=[.!?])\s+", text.strip())

    return [s.strip() for s in sentences if s.strip()]


# --------------------------------------------------
# STEP 4: Finite-state sequence modeling
# --------------------------------------------------

def separate_documents(text):

    sentences = split_sentences(text)

    if not sentences:
        return [], []

    # Predict topic of every sentence
    predictions = model.predict(sentences)

    probabilities = model.predict_proba(sentences)

    documents = []

    # Initial state
    current_document = [sentences[0]]

    current_state = predictions[0]

    details = []

    details.append({
        "sentence": sentences[0],
        "topic": predictions[0],
        "confidence": max(probabilities[0])
    })

    # Process remaining sentences
    for i in range(1, len(sentences)):

        sentence = sentences[i]

        predicted_topic = predictions[i]

        details.append({
            "sentence": sentence,
            "topic": predicted_topic,
            "confidence": max(probabilities[i])
        })

        # State transition
        if predicted_topic != current_state:

            # Save previous document
            documents.append(" ".join(current_document))

            # Start a new document
            current_document = [sentence]

            current_state = predicted_topic

        else:

            # Continue current document
            current_document.append(sentence)

    # Add final document
    if current_document:

        documents.append(" ".join(current_document))

    return documents, details


# --------------------------------------------------
# STEP 5: Input text
# --------------------------------------------------

text = """
Machine learning uses data to make predictions.
Neural networks are used in deep learning.

The cricket team won the match yesterday.
The player scored a century in the match.

Python is a popular programming language.
Python supports object oriented programming.
"""


# --------------------------------------------------
# STEP 6: Run document separation
# --------------------------------------------------

documents, details = separate_documents(text)


# --------------------------------------------------
# STEP 7: Display classification results
# --------------------------------------------------

print("\nSENTENCE CLASSIFICATION RESULTS")

print("-" * 60)

for item in details:

    print("Sentence:", item["sentence"])

    print("Predicted topic:", item["topic"])

    print("Confidence:", round(item["confidence"], 3))

    print()


# --------------------------------------------------
# STEP 8: Display separated documents
# --------------------------------------------------

print("\nSEPARATED DOCUMENTS")

print("-" * 60)

for i, document in enumerate(documents, start=1):

    print("\nDocument", i, ":")

    print(document)


# --------------------------------------------------
# STEP 9: Analyze results
# --------------------------------------------------

print("\nRESULT ANALYSIS")

print("-" * 60)

print("Total sentences:", len(details))

print("Total documents:", len(documents))

topic_distribution = Counter(
    item["topic"] for item in details
)

print("Topic distribution:", dict(topic_distribution))


# --------------------------------------------------
# STEP 10: Evaluate classifier
# --------------------------------------------------

training_predictions = model.predict(training_sentences)

accuracy = accuracy_score(
    training_labels,
    training_predictions
)

print(
    "\nTraining classification accuracy:",
    round(accuracy * 100, 2),
    "%"
)



SENTENCE CLASSIFICATION RESULTS
------------------------------------------------------------
Sentence: Machine learning uses data to make predictions.
Predicted topic: ML
Confidence: 0.994

Sentence: Neural networks are used in deep learning.
Predicted topic: ML
Confidence: 0.958

Sentence: The cricket team won the match yesterday.
Predicted topic: CRICKET
Confidence: 0.997

Sentence: The player scored a century in the match.
Predicted topic: CRICKET
Confidence: 0.994

Sentence: Python is a popular programming language.
Predicted topic: PROGRAMMING
Confidence: 0.988

Sentence: Python supports object oriented programming.
Predicted topic: PROGRAMMING
Confidence: 0.99


SEPARATED DOCUMENTS
------------------------------------------------------------

Document 1 :
Machine learning uses data to make predictions. Neural networks are used in deep learning.

Document 2 :
The cricket team won the match yesterday. The player scored a century in the match.

Document 3 :
Python is a popular prog